In [1]:
# Cell 1 – Imports & basic setup
import pandas as pd
import os
from datasets import load_dataset
from langdetect import detect
from transformers import pipeline
from pyabsa import AspectTermExtraction as ATEPC
import warnings

warnings.filterwarnings("ignore", category=UserWarning)       # reduces tokenizer noise
pd.set_option('display.max_colwidth', 150)                   # better display of long reviews

print("Imports done")

No CUDA GPU found in your device
[2026-01-20 20:05:19] (2.4.2) PyABSA(2.4.2): If your code crashes on Colab, please use the GPU runtime. Then run "pip install pyabsa[dev] -U" and restart the kernel.
Or if it does not work, you can use v1.x versions, e.g., pip install pyabsa<2.0 -U




Try to downgrade transformers<=4.29.0.




Imports done


In [2]:
# Cell 2 – Load original dataset + downsample to 1000 rows
dataset_name = "farhanenzo/Customer_Reviews_Dataset_Online_Food_Ordering_Portal_Bangladesh"

dataset = load_dataset(dataset_name)
df_full = dataset['train'].to_pandas()

# keep only useful columns
df_full = df_full[['reviewer_name', 'review_text', 'ratings_int']]
df_full.rename(columns={
    'reviewer_name': 'customer_name',
    'review_text': 'review',
    'ratings_int': 'rating'
}, inplace=True)

# Downsample – random 1000 rows (fixed seed = reproducible)
df = df_full.sample(n=1000, random_state=42).reset_index(drop=True)

print(f"Original dataset size: {len(df_full):,d} rows")
print(f"Working with: {len(df):,} randomly selected rows")
print(df.head(3))

Original dataset size: 43,061 rows
Working with: 1,000 randomly selected rows
  customer_name                                         review  rating
0        Tasfia                                       very bad       1
1     Professor  সেরা চিকেন চাপ, সাইজ বড় ছিল।দাম অনুযায়ী জোস🌟👌       5
2            MD                                            bad       2


In [3]:
# Cell 3 – Translation with checkpoint
translation_file = 'df_1000_translated.parquet'

if os.path.exists(translation_file):
    df = pd.read_parquet(translation_file)
    print(f"Loaded existing translated data → {len(df):,} rows")
else:
    print("Translating 1000 reviews... (should take 5–20 minutes)")
    
    translator = pipeline('translation', model='Helsinki-NLP/opus-mt-bn-en')
    
    def translate_to_english(text):
        try:
            lang = detect(text)
            if lang != 'en':
                return translator(text, max_length=512)[0]['translation_text']
            return text
        except Exception as e:
            # print(f"Translation error: {e} → keeping original")
            return text
    
    df['review_en'] = df['review'].apply(translate_to_english)
    
    # save immediately
    df.to_parquet(translation_file, index=False)
    print(f"Translation finished → saved to {translation_file}")
    
print("\nTranslation sample:")
print(df[['review', 'review_en']].head(6))

Loaded existing translated data → 1,000 rows

Translation sample:
                                                                                                     review  \
0                                                                                                  very bad   
1                                                             সেরা চিকেন চাপ, সাইজ বড় ছিল।দাম অনুযায়ী জোস🌟👌   
2                                                                                                       bad   
3                                                               It was disappointing burnt pieces of mutton   
4                                                                                     burger ta moja chilo💖   
5  পোরা পোরা গন্ধ পাচ্ছিলাম যার কারনে খাইতেই পারিনাই। শুধু হাড় আর তেল দিয়েই ভর্তি! মাংশ তেমন নাই বললেই চলে!   

                                                              review_en  
0                                                             Himy Bard  
1       

In [4]:
# Cell 4 – Load PyABSA model from your local checkpoint
checkpoint_dir = r"C:\Users\Shabeel\Desktop\bkash-review-analysis\checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43"   # ← PASTE YOUR ACTUAL PATH HERE

print("Loading model from:", checkpoint_dir)
print("Files in folder:", os.listdir(checkpoint_dir))

aspect_extractor = ATEPC.AspectExtractor(
    checkpoint=checkpoint_dir,
    auto_device=True
)

print("Aspect extractor loaded")

Loading model from: C:\Users\Shabeel\Desktop\bkash-review-analysis\checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43
Files in folder: ['fast_lcf_atepc.args.txt', 'fast_lcf_atepc.config', 'fast_lcf_atepc.state_dict', 'fast_lcf_atepc.tokenizer']
[2026-01-20 20:05:30] (2.4.2) Load aspect extractor from C:\Users\Shabeel\Desktop\bkash-review-analysis\checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43
[2026-01-20 20:05:30] (2.4.2) config: checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43\fast_lcf_atepc.config
[2026-01-20 20:05:30] (2.4.2) state_dict: checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.89_atef1_75.43\fast_lcf_atepc.state_dict
[2026-01-20 20:05:30] (2.4.2) model: None
[2026-01-20 20:05:30] (2.4.2) tokenizer: checkpoints\ATEPC_ENGLISH_CHECKPOINT\fast_lcf_atepc_English_cdw_apcacc_82.36_apcf1_81.8

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


Aspect extractor loaded


In [5]:
# Cell 5 – Define analysis function
def analyze_review(text):
    if not text or len(text.strip()) < 5:
        return []
        
    result = aspect_extractor.predict(
        text,
        print_result=False,
        ignore_error=True
    )
    
    aspects = []
    for i, asp in enumerate(result.get('aspect', [])):
        sent = result['sentiment'][i]
        if sent in ['Positive', 'Negative']:
            aspects.append({
                'focus_point': asp.lower().strip(),
                'sentiment': 'positive' if sent == 'Positive' else 'negative'
            })
    return aspects

In [6]:
# Cell 6 – Run analysis + partial saving
partial_file = 'partial_analysis_1000.csv'

# Load existing partial results if any
if os.path.exists(partial_file):
    df_partial = pd.read_csv(partial_file)
    results = df_partial.to_dict('records')
    processed_reviews = set(df_partial['review'].astype(str))
    print(f"Resuming → already processed {len(processed_reviews)} reviews")
else:
    results = []
    processed_reviews = set()
    print("Starting analysis from scratch")

print(f"Total rows to process: {len(df)}")

for idx, row in df.iterrows():
    if row['review'] in processed_reviews:
        continue
        
    aspects = analyze_review(row['review_en'])
    
    for asp in aspects:
        results.append({
            'customer_name': row['customer_name'],
            'review': row['review'],
            'focus_point': asp['focus_point'],
            'sentiment': asp['sentiment']
        })
    
    # save every 200 processed reviews
    if len(results) % 200 == 0 and len(results) > 0:
        pd.DataFrame(results).to_csv(partial_file, index=False)
        print(f"Checkpoint saved → {len(results)} aspect rows")

# Final save
df_output = pd.DataFrame(results)

if not df_output.empty:
    df_output['focus_point_count'] = df_output.groupby(['focus_point', 'sentiment'])['focus_point'].transform('count')
    df_output = df_output.sort_values('focus_point_count', ascending=False)
    
    final_file = 'final_analysis_1000.csv'
    df_output.to_csv(final_file, index=False)
    print(f"\nFinal results saved to {final_file}")
    print(f"Total aspect rows: {len(df_output):,d}")
    
    # Show top focus points
    summary = df_output.groupby(['focus_point', 'sentiment'])['focus_point_count'].max().reset_index()
    summary = summary.sort_values('focus_point_count', ascending=False)
    print("\nTop 15 focus points:")
    print(summary.head(15))
else:
    print("No aspects were extracted – check translation quality or model behavior")

Resuming → already processed 409 reviews
Total rows to process: 1000
[2026-01-20 20:06:46] (2.4.2) The results of aspect term extraction have been saved in c:\Users\Shabeel\Desktop\bkash-review-analysis\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
Checkpoint saved → 600 aspect rows
Checkpoint saved → 600 aspect rows
[2026-01-20 20:06:55] (2.4.2) The results of aspect term extraction have been saved in c:\Users\Shabeel\Desktop\bkash-review-analysis\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
Checkpoint saved → 600 aspect rows
[2026-01-20 20:06:58] (2.4.2) The results of aspect term extraction have been saved in c:\Users\Shabeel\Desktop\bkash-review-analysis\Aspect Term Extraction and Polarity Classification.FAST_LCF_ATEPC.result.json
Checkpoint saved → 600 aspect rows
[2026-01-20 20:07:02] (2.4.2) The results of aspect term extraction have been saved in c:\Users\Shabeel\Desktop\bkash-review-analysis\Aspect Term Extractio

In [7]:
df_output.tail(10)

,customer_name,review,focus_point,sentiment,focus_point_count
679,T,Packing not well done. Doughnuts were all over the place inside the package. Cream that was supposed to be on top of the doughnut was smeared on t...,cream,negative,1.0
311,Joy,"french fry was not good ( baje rokomer jhal chilo) kintu burger gulo khubi valo chilo, testy chilo.",french fry,negative,1.0
309,Humaira,mashallah food is marvelous,mashallah food,positive,1.0
289,Fatimah,Food quantity was extremely little compared to earlier. Also it was a little burnt. Also delivery time was a lot.,food quantity,negative,1.0
287,Foysal,"৫স্টার দিলাম, যদিও কাচ্চিতে মাংসের চেয়ে হাড় বেশি ছিলো।",bone,negative,1.0
39,Rosemila,"The nachos were aweful,however the pasta tasted good.",nachos,negative,1.0
38,Nishat,ফালতু ছিল।\n৪৫ মিনিট অপেক্ষা করেছি অর্ডার দিয়ে। খুব বাজে ভবে সময় নষ্ট হয়েছে।খাবার টা খুব উৎকৃষ্ট ছিলো না।,time,negative,1.0
711,Shihab,The behave o delivery man was to bad.I told him to bring the change but.......,delivery man,negative,1.0
295,Khaled,"Good shik, nan and porata. But the salad dressing had bad smell.",NaN,positive,NaN
543,Sajib,"halim test was amazing 😋and also luchu , nan and Monglai test was good",NaN,positive,NaN


In [8]:
!pip install pandas-gbq google-auth google-auth-oauthlib google-auth-httplib2 pydata-google-auth

c:\Users\Shabeel\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=3>
  res = process_handler(cmd, _system_body)
c:\Users\Shabeel\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=4>
  res = process_handler(cmd, _system_body)
c:\Users\Shabeel\AppData\Local\Programs\Python\Python312\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=5>
  res = process_handler(cmd, _system_body)


In [10]:
import pandas_gbq

project_id = "food-review-sentiment-analysis"
dataset_id = "foodpanda_reviews"
table_id = "analyzed_reviews_1000"

full_table_id = f"{project_id}.{dataset_id}.{table_id}"

pandas_gbq.to_gbq(
    df_output, 
    full_table_id,
    project_id=project_id,
    if_exists='replace'
)

c:\Users\Shabeel\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas_gbq\gbq_connector.py:319: PendingDeprecationWarning: In a future major release, the default delimiter will be changed to a `/` in accordance with RFC9110.
  user_agent = create_user_agent(
c:\Users\Shabeel\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas_gbq\gbq_connector.py:319: PendingDeprecationWarning: In a future major release, the default delimiter will be changed to a `/` in accordance with RFC9110.
  user_agent = create_user_agent(
100%|██████████| 1/1 [00:00<?, ?it/s]
